# KDD Cup 2015 — Raw Event Extraction, Scaling & K-Means

This version is prepared for **Google Colab with Google Drive**.

Expected data structure inside Google Drive:

```text
MyDrive/
└── KDD_Cup_2015/
    └── data/
        └── raw/
            └── train/
                └── train/
                    └── log_train.csv
```

**Important:** If your folder name or dataset location is different, change only `BASE_DIR` in the next cell.

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

print('Google Drive mounted successfully.')


In [ ]:
# ============================================================
# CELL 2 — Set project paths
# ============================================================
import os

# CHANGE THIS ONE LINE if your KDD Cup folder has another name/location.
BASE_DIR = '/content/drive/MyDrive/KDD_Cup_2015'

RAW_LOG_PATH = os.path.join(
    BASE_DIR, 'data', 'raw', 'train', 'train', 'log_train.csv'
)

OUTPUT_DIR = os.path.join(BASE_DIR, 'data', 'temporal_event_counts')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Project directory:', BASE_DIR)
print('Raw log file:', RAW_LOG_PATH)
print('Output directory:', OUTPUT_DIR)

if not os.path.exists(RAW_LOG_PATH):
    raise FileNotFoundError(
        f'log_train.csv was not found at:\n{RAW_LOG_PATH}\n\n'
        'Check BASE_DIR and the folder structure in Google Drive.'
    )

print('✓ log_train.csv found.')


In [ ]:
# ============================================================
# CELL 3 — Import libraries and load KDD Cup 2015 logs
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print('Loading KDD Cup 2015 log data...')

log_train = pd.read_csv(RAW_LOG_PATH)
log_train['time'] = pd.to_datetime(log_train['time'])
log_train = log_train.sort_values(['enrollment_id', 'time']).reset_index(drop=True)

print('Shape:', log_train.shape)
print('Columns:', log_train.columns.tolist())
display(log_train.head())


In [ ]:
# ============================================================
# CELL 4 — Extract temporal event-count features and normalize
# ============================================================
print('Calculating temporal slice masks...')

first_time = log_train.groupby('enrollment_id')['time'].transform('min')
last_time = log_train.groupby('enrollment_id')['time'].transform('max')

w1_mask = (
    (log_train['time'] - first_time).dt.total_seconds()
    <= 7 * 24 * 3600
)

last_mask = (
    (last_time - log_train['time']).dt.total_seconds()
    <= 7 * 24 * 3600
)

slices = {
    'full': log_train,
    'w1': log_train[w1_mask],
    'last': log_train[last_mask]
}

master_cols = None

for name, subset in slices.items():
    print(f'Processing {name} temporal slice...')

    # Count each event type for every enrollment/student.
    df_slice = (
        subset.groupby(['enrollment_id', 'event'])
        .size()
        .unstack(fill_value=0)
    )

    df_slice.columns = [f'event_{col}' for col in df_slice.columns]
    df_slice = df_slice.reset_index()

    # Keep exactly the same event-feature columns across all slices.
    if master_cols is None:
        master_cols = df_slice.columns
    else:
        df_slice = df_slice.reindex(columns=master_cols, fill_value=0)

    feat_cols = [c for c in master_cols if c != 'enrollment_id']

    # Standardization: mean = 0, standard deviation = 1.
    scaler = StandardScaler()
    df_slice[feat_cols] = scaler.fit_transform(df_slice[feat_cols])

    out_path = os.path.join(OUTPUT_DIR, f'{name}_norm.csv')
    df_slice.to_csv(out_path, index=False)

    print(f'Saved {name} shape: {df_slice.shape}')
    print(f'Path: {out_path}\n')

print('✓ Temporal event extraction and scaling completed.')


In [ ]:
# ============================================================
# CELL 5 — Check generated files
# ============================================================
print('Generated files:')
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(' -', filename)

display(pd.read_csv(os.path.join(OUTPUT_DIR, 'full_norm.csv')).head())


In [ ]:
# ============================================================
# CELL 6 — GPU check (optional)
# ============================================================
import torch

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(f'Executing acceleration on: {device}')

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Enable a GPU from Runtime > Change runtime type if needed.')


In [ ]:
# ============================================================
# CELL 7 — K-Means clustering + evaluation + PCA visualization
# ============================================================
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

input_cols = [
    'event_video',
    'event_access',
    'event_wiki',
    'event_discussion'
]

slice_names = ['full', 'w1', 'last']

log_train = pd.read_csv(RAW_LOG_PATH)
log_train['time'] = pd.to_datetime(log_train['time'])
log_train = log_train.sort_values(['enrollment_id', 'time'])

first_time = log_train.groupby('enrollment_id')['time'].transform('min')
last_time = log_train.groupby('enrollment_id')['time'].transform('max')

w1_mask = (
    (log_train['time'] - first_time).dt.total_seconds()
    <= 7 * 24 * 3600
)

last_mask = (
    (last_time - log_train['time']).dt.total_seconds()
    <= 7 * 24 * 3600
)

slice_dataframes = {
    'full': log_train,
    'w1': log_train[w1_mask],
    'last': log_train[last_mask]
}

all_results = {}

for slice_name in slice_names:

    norm_path = os.path.join(
        OUTPUT_DIR,
        f'{slice_name}_norm.csv'
    )

    if not os.path.exists(norm_path):
        print(f'Skipping {slice_name}: file not found.')
        continue

    df_norm = pd.read_csv(norm_path)

    cols_to_use = [
        c for c in input_cols
        if c in df_norm.columns
    ]

    if len(cols_to_use) == 0:
        print(f'Skipping {slice_name}: no requested event columns found.')
        continue

    X_np = df_norm[cols_to_use].values

    # K-Means clustering: K = 3
    kmeans = KMeans(
        n_clusters=3,
        random_state=42,
        n_init=10
    )

    clusters = kmeans.fit_predict(X_np)
    df_norm['cluster'] = clusters

    # Evaluation metrics
    sil = silhouette_score(X_np, clusters)
    db = davies_bouldin_score(X_np, clusters)
    ch = calinski_harabasz_score(X_np, clusters)

    print('\n' + '=' * 60)
    print(f'INPUT DIMENSION: {slice_name.upper()}')
    print('=' * 60)
    print(f'Silhouette: {sil:.3f}')
    print(f'Davies-Bouldin: {db:.3f}')
    print(f'Calinski-Harabasz: {ch:.1f}')

    # Average RAW event counts per cluster
    subset_df = slice_dataframes[slice_name]

    raw_counts = (
        subset_df
        .groupby(['enrollment_id', 'event'])
        .size()
        .unstack(fill_value=0)
    )

    raw_counts.columns = [
        f'event_{col}'
        for col in raw_counts.columns
    ]

    raw_counts = raw_counts.reset_index()

    merged_raw = (
        df_norm[['enrollment_id', 'cluster']]
        .merge(
            raw_counts,
            on='enrollment_id',
            how='left'
        )
        .fillna(0)
    )

    raw_cols_to_use = [
        c for c in input_cols
        if c in merged_raw.columns
    ]

    raw_means = (
        merged_raw
        .groupby('cluster')[raw_cols_to_use]
        .mean()
    )

    print('\nAverage Raw Event Counts per Cluster:')
    print(raw_means.round(2).to_string())

    # PCA 2-D projection
    pca = PCA(
        n_components=2,
        random_state=42
    )

    X_pca = pca.fit_transform(X_np)

    df_norm['pca1'] = X_pca[:, 0]
    df_norm['pca2'] = X_pca[:, 1]

    pca_centroids = pca.transform(
        kmeans.cluster_centers_
    )

    # Plot
    plt.figure(figsize=(8, 6))

    scatter = plt.scatter(
        df_norm['pca1'],
        df_norm['pca2'],
        c=df_norm['cluster'],
        cmap='tab10',
        alpha=0.5,
        s=25
    )

    plt.scatter(
        pca_centroids[:, 0],
        pca_centroids[:, 1],
        s=250,
        c='black',
        marker='X',
        label='Centroids'
    )

    plt.title(
        f'Input Dimension Clusters (K=3) - {slice_name.upper()}'
    )
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.colorbar(
        scatter,
        label='Cluster ID'
    )
    plt.legend()
    plt.grid(
        True,
        linestyle='--',
        alpha=0.5
    )
    plt.tight_layout()
    plt.show()

    # Save clustering results to Google Drive
    cluster_path = os.path.join(
        OUTPUT_DIR,
        f'{slice_name}_clusters.csv'
    )

    df_norm.to_csv(
        cluster_path,
        index=False
    )

    print(
        f'Clustered data saved to: {cluster_path}'
    )

    all_results[slice_name] = {
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski_harabasz': ch,
        'cluster_means_raw': raw_means
    }

print('\n✓ K-Means analysis completed.')


In [ ]:
# ============================================================
# CELL 8 — Compare clustering metrics
# ============================================================
summary_rows = []

for name, result in all_results.items():
    summary_rows.append({
        'Temporal Slice': name,
        'Silhouette': result['silhouette'],
        'Davies-Bouldin': result['davies_bouldin'],
        'Calinski-Harabasz': result['calinski_harabasz']
    })

metrics_df = pd.DataFrame(summary_rows)

display(metrics_df.round(3))

print('All outputs are stored in:')
print(OUTPUT_DIR)
